In [1]:
import os
import time
import ctypes
import numpy as np
import pyxrt
import subprocess
import pandas as pd
import re

# Compiling the C code reference with optimization flags

In [2]:
cpu_funcs = {}

threads_configs = [1, 2, 4]

for threads in threads_configs:
    c_file = f"mmult_ref_openmp_{threads}.c"
    so_file = f"mmult_ref_openmp_{threads}.so"
    
    subprocess.run([
        "gcc", "-O3", "-fopenmp", "-march=native",
        "-shared", "-fPIC", c_file, "-o", so_file
    ], check=True)
    
    libc = ctypes.CDLL(f"./{so_file}")
    func = libc.matmul_large_cpu
    func.argtypes = [
        ctypes.POINTER(ctypes.c_uint32), 
        ctypes.POINTER(ctypes.c_uint32), 
        ctypes.POINTER(ctypes.c_uint32), 
        ctypes.c_int
    ]
    
    cpu_funcs[threads] = func

# Initializing FPGA

In [3]:
XCLBIN_PATH = "krnl_mult.xclbin"
BLOCK_SIZE = 128  # Hardware block size
BUFFER_SIZE_BYTES = BLOCK_SIZE * BLOCK_SIZE * 4

device = pyxrt.device(0)
uuid = device.load_xclbin(XCLBIN_PATH)

# Instantiating the 4 independent Compute Units configured in the .cfg file
cus = []
cu_buffers = []

for i in range(1, 5):
    cu_name = f"krnl_mult:{{krnl_mult_{i}}}"
    kernel_instance = pyxrt.kernel(device, uuid, cu_name)
    cus.append(kernel_instance)
    
    # Allocation of specific buffers in the correct memory groups (HBM mapped in .cfg)
    bo_a = pyxrt.bo(device, BUFFER_SIZE_BYTES, pyxrt.bo.flags.normal, kernel_instance.group_id(0))
    bo_b = pyxrt.bo(device, BUFFER_SIZE_BYTES, pyxrt.bo.flags.normal, kernel_instance.group_id(1))
    bo_out = pyxrt.bo(device, BUFFER_SIZE_BYTES, pyxrt.bo.flags.normal, kernel_instance.group_id(2))
    
    cu_buffers.append({'a': bo_a, 'b': bo_b, 'out': bo_out})

print(f"Success: 4 CUs initialized and mapped to their respective HBM banks")


Success: 4 CUs initialized and mapped to their respective HBM banks


# Auxiliary Function for CUs treatment

In [4]:
def matmul_fpga_multi_cu(A, B, N, num_cus=1):
    C = np.zeros((N, N), dtype=np.uint32)
    
    if N < BLOCK_SIZE:
        sub_A = np.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=np.uint32)
        sub_B = np.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=np.uint32)
        sub_A[:N, :N] = A
        sub_B[:N, :N] = B
        
        cu_id = 0
        cu_buffers[cu_id]['a'].write(np.ascontiguousarray(sub_A).flatten(), 0)
        cu_buffers[cu_id]['b'].write(np.ascontiguousarray(sub_B).flatten(), 0)
        
        cu_buffers[cu_id]['a'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE)
        cu_buffers[cu_id]['b'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE)
        
        run = cus[cu_id](cu_buffers[cu_id]['a'], cu_buffers[cu_id]['b'], cu_buffers[cu_id]['out'], BLOCK_SIZE)
        run.wait()
        
        cu_buffers[cu_id]['out'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_FROM_DEVICE)
        raw_bytes = cu_buffers[cu_id]['out'].read(BUFFER_SIZE_BYTES, 0)
        res_128 = raw_bytes.view(np.uint32).reshape((BLOCK_SIZE, BLOCK_SIZE))
        
        C[:N, :N] = res_128[:N, :N]
        return C

    num_blocks = N // BLOCK_SIZE
    block_operations = []
    for i in range(num_blocks):
        for j in range(num_blocks):
            for k in range(num_blocks):
                block_operations.append((i, j, k))
                
    for idx in range(0, len(block_operations), num_cus):
        current_batch = block_operations[idx:idx+num_cus]
        active_runs = []
        
        for cu_id, (i, j, k) in enumerate(current_batch):
            sub_A = np.ascontiguousarray(A[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE, k*BLOCK_SIZE:(k+1)*BLOCK_SIZE])
            sub_B = np.ascontiguousarray(B[k*BLOCK_SIZE:(k+1)*BLOCK_SIZE, j*BLOCK_SIZE:(j+1)*BLOCK_SIZE])
            
            cu_buffers[cu_id]['a'].write(sub_A.flatten(), 0)
            cu_buffers[cu_id]['b'].write(sub_B.flatten(), 0)
            cu_buffers[cu_id]['a'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE)
            cu_buffers[cu_id]['b'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE)
            
            run = cus[cu_id](cu_buffers[cu_id]['a'], cu_buffers[cu_id]['b'], cu_buffers[cu_id]['out'], BLOCK_SIZE)
            active_runs.append((run, cu_id, i, j))
            
        for run, cu_id, i, j in active_runs:
            run.wait()
            cu_buffers[cu_id]['out'].sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_FROM_DEVICE)
            raw_bytes = cu_buffers[cu_id]['out'].read(BUFFER_SIZE_BYTES, 0)
            sub_C_res = raw_bytes.view(np.uint32).reshape((BLOCK_SIZE, BLOCK_SIZE))
            C[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE, j*BLOCK_SIZE:(j+1)*BLOCK_SIZE] += sub_C_res
            
    return C

# Benchmarking and Functional Verification

In [5]:
sizes = [4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
num_runs = 5

results_table = []

for N in sizes:
    print(f"\n Matrix {N}x{N}")
    
    cu_configs = [1]
    if N >= 256:
        cu_configs.append(2)
    if N >= 512:
        cu_configs.append(4)
        
    cpu_configs = [1, 2, 4]
    
    total_cpu_ms = {t: 0.0 for t in cpu_configs}
    total_fpga_ms = {cus: 0.0 for cus in cu_configs}
    correct_all = {cus: True for cus in cu_configs}
    
    for run in range(num_runs):
        A_large = np.random.randint(1, 5, size=(N, N), dtype=np.uint32)
        B_large = np.random.randint(1, 5, size=(N, N), dtype=np.uint32)
        
        A_ptr = A_large.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))
        B_ptr = B_large.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))
        
        C_cpu_ref = np.zeros((N, N), dtype=np.uint32)
        
        for threads in cpu_configs:
            C_cpu = np.zeros((N, N), dtype=np.uint32)
            C_ptr = C_cpu.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))
            
            t_start_cpu = time.perf_counter()
            cpu_funcs[threads](A_ptr, B_ptr, C_ptr, N)
            t_end_cpu = time.perf_counter()
            
            total_cpu_ms[threads] += (t_end_cpu - t_start_cpu) * 1000
            
            if threads == 1:
                C_cpu_ref = C_cpu.copy()
        
        for num_cus in cu_configs:
            t_start_fpga = time.perf_counter()
            C_fpga = matmul_fpga_multi_cu(A_large, B_large, N, num_cus=num_cus)
            t_end_fpga = time.perf_counter()
            
            total_fpga_ms[num_cus] += (t_end_fpga - t_start_fpga) * 1000
            
            if not np.array_equal(C_cpu_ref, C_fpga):
                correct_all[num_cus] = False
                
    for num_cus in cu_configs:
        avg_fpga_ms = total_fpga_ms[num_cus] / num_runs
        
        for threads in cpu_configs:
            avg_cpu_ms = total_cpu_ms[threads] / num_runs
            speedup = avg_cpu_ms / avg_fpga_ms if avg_fpga_ms > 0 else 0
            
            results_table.append({
                'Size': f"{N}x{N}",
                'FPGA CUs': num_cus,
                'CPU Threads': threads,
                'Avg CPU (ms)': avg_cpu_ms,
                'Avg FPGA (ms)': avg_fpga_ms,
                'Speedup (x)': speedup,
                'Status': 'Passed' if correct_all[num_cus] else 'Failed'
            })

for buf in cu_buffers:
    del buf['a'], buf['b'], buf['out']
del cus, device


 Matrix 4x4

 Matrix 8x8

 Matrix 16x16

 Matrix 32x32

 Matrix 64x64

 Matrix 128x128

 Matrix 256x256

 Matrix 512x512

 Matrix 1024x1024

 Matrix 2048x2048

 Matrix 4096x4096


In [6]:
df_results = pd.DataFrame(results_table)
df_results

,Size,FPGA CUs,CPU Threads,Avg CPU (ms),Avg FPGA (ms),Speedup (x),Status
0,4x4,1,1,0.021813,1.113842,0.019583,Passed
1,4x4,1,2,0.072081,1.113842,0.064714,Passed
2,4x4,1,4,0.053678,1.113842,0.048192,Passed
3,8x8,1,1,0.005015,1.028885,0.004874,Passed
4,8x8,1,2,0.003850,1.028885,0.003742,Passed
5,8x8,1,4,0.034139,1.028885,0.033180,Passed
6,16x16,1,1,0.006203,1.033206,0.006004,Passed
7,16x16,1,2,0.006394,1.033206,0.006188,Passed
8,16x16,1,4,0.032346,1.033206,0.031306,Passed
9,32x32,1,1,0.026957,1.039326,0.025937,Passed


# Energy Consumption Measurement

In [6]:
RAPL_ENERGY_FILE = "/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj"

def read_rapl_uj():
    with open(RAPL_ENERGY_FILE, "r") as f:
        return int(f.read().strip())

def measure_energy_rapl(threads: int, matrix_size: int = 128, num_runs: int = 5) -> dict:
    so_file = f"./mmult_ref_openmp_{threads}.so"
    libc = ctypes.CDLL(so_file)
    func = libc.matmul_large_cpu
    func.argtypes = [
        ctypes.POINTER(ctypes.c_uint32),
        ctypes.POINTER(ctypes.c_uint32),
        ctypes.POINTER(ctypes.c_uint32),
        ctypes.c_int
    ]

    energies_j = []
    powers_w   = []
    N = matrix_size

    for _ in range(num_runs):
        A = np.random.randint(1, 5, (N, N), dtype=np.uint32)
        B = np.random.randint(1, 5, (N, N), dtype=np.uint32)
        C = np.zeros((N, N), dtype=np.uint32)
        A_ptr = A.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))
        B_ptr = B.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))
        C_ptr = C.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))

        e_start = read_rapl_uj()
        t_start = time.perf_counter()

        func(A_ptr, B_ptr, C_ptr, N)

        t_end = time.perf_counter()
        e_end = read_rapl_uj()

        delta_j = (e_end - e_start) * 1e-6 
        delta_s = t_end - t_start
        energies_j.append(delta_j)
        powers_w.append(delta_j / delta_s if delta_s > 0 else None)

    e = np.array(energies_j)
    w = np.array([x for x in powers_w if x is not None])
    return {
        "threads":     threads,
        "matrix_size": matrix_size,
        "mean_joules": float(e.mean()),
        "std_joules":  float(e.std()),
        "mean_watts":  float(w.mean()) if len(w) else None,
        "std_watts":   float(w.std())  if len(w) else None,
        "runs":        len(e),
    }


MATRIX_SIZE = 128
NUM_RUNS    = 5

print(f"Measuring CPU energy & power via RAPL sysfs — {MATRIX_SIZE}x{MATRIX_SIZE}, {NUM_RUNS} runs\n")

energy_results = []
for t in [1, 2, 4]:
    print(f"  Running with {t} thread(s)", end=" ", flush=True)
    res = measure_energy_rapl(threads=t, matrix_size=MATRIX_SIZE, num_runs=NUM_RUNS)
    energy_results.append(res)
    print(f"mean = {res['mean_joules']:.4f} J  |  {res['mean_watts']:.3f} W  ")

df_energy = pd.DataFrame([
    {
        "CPU Threads"    : r["threads"],
        "Matrix Size"    : f"{r['matrix_size']}x{r['matrix_size']}",
        "Mean Power (W)" : round(r["mean_watts"],  4) if r["mean_watts"] else None
    }
    for r in energy_results
])

display(df_energy)

Measuring CPU energy & power via RAPL sysfs — 128x128, 5 runs

  Running with 1 thread(s) mean = 0.0511 J  |  29.374 W  
  Running with 2 thread(s) mean = 0.0394 J  |  55.372 W  
  Running with 4 thread(s) mean = 0.0303 J  |  41.284 W  


,CPU Threads,Matrix Size,Mean Power (W)
0,1,128x128,29.3737
1,2,128x128,55.3715
2,4,128x128,41.2844
